In [1]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
import openai
import anthropic
from google import genai
from google.genai import types
import mlflow
from rouge_score import rouge_scorer

load_dotenv()

if "GOOGLE_API_KEY" in os.environ:
    os.environ.pop("GOOGLE_API_KEY")

openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Loading CNN/DailyMail dataset...")
dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")
samples = dataset.select(range(100))

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

SYSTEM_PROMPT = "You are a news summarizer. Summarize the article in 2-3 sentences. Return only the summary, no preamble."

MAX_RETRIES = 3  # retry transient API errors (rate limits, 5xx, timeouts) before giving up on a sample

def summarize_openai(article):
    last_exc = None
    for attempt in range(MAX_RETRIES):
        start = time.time()
        try:
            response = openai_client.chat.completions.create(
                model="gpt-4o",  # FIX
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": article[:3000]}
                ],
                max_tokens=150,
                temperature=0.0  # FIX
            )
            return response.choices[0].message.content.strip(), time.time() - start
        except Exception as e:
            last_exc = e
            if attempt < MAX_RETRIES - 1:
                wait = (attempt + 1) * 3
                print(f"OpenAI error, retrying in {wait}s (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
                time.sleep(wait)
    print(f"OpenAI error after retries: {last_exc}")
    return "", time.time() - start

def summarize_anthropic(article):
    last_exc = None
    for attempt in range(MAX_RETRIES):
        start = time.time()
        try:
            response = anthropic_client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=150,
                temperature=0.0,  # FIX: match GPT-4o/Gemini's greedy decoding for a fair comparison
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": article[:3000]}]
            )
            return response.content[0].text.strip(), time.time() - start
        except Exception as e:
            last_exc = e
            if attempt < MAX_RETRIES - 1:
                wait = (attempt + 1) * 3
                print(f"Anthropic error, retrying in {wait}s (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
                time.sleep(wait)
    print(f"Anthropic error after retries: {last_exc}")
    return "", time.time() - start

def summarize_gemini(article):
    for attempt in range(4):
        start = time.time()  # FIX: reset per attempt so retry backoff sleep isn't counted as latency
        try:
            response = gemini_client.models.generate_content(
                model="gemini-2.5-flash",
                contents=f"{SYSTEM_PROMPT}\n\n{article[:3000]}",
                config=types.GenerateContentConfig(
                    max_output_tokens=150,
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(thinking_budget=0)
                )
            )
            return response.text.strip() if response.text else "", time.time() - start
        except Exception as e:
            if "503" in str(e) and attempt < 3:
                time.sleep((attempt + 1) * 3)
                continue
            print(f"Gemini error: {e}")
            return "", time.time() - start

def compute_rouge(hypothesis, reference):
    scores = scorer.score(reference, hypothesis)
    return scores["rouge1"].fmeasure, scores["rouge2"].fmeasure, scores["rougeL"].fmeasure

# Fixed (non-timestamped) path so an interrupted run can resume instead of
# losing already-paid-for API calls.
os.makedirs("../logs", exist_ok=True)
csv_path = "../logs/cnn_dailymail_baseline.csv"

if os.path.exists(csv_path):
    results = pd.read_csv(csv_path).to_dict("records")
    print(f"Resuming: {len(results)}/{len(samples)} samples already done")
else:
    results = []

mlflow.set_experiment("cnn_dailymail_baseline")

with mlflow.start_run(run_name="summarization_baseline_v2_fixed"):
    mlflow.log_param("sample_count", 100)
    mlflow.log_param("openai_model", "gpt-4o")
    mlflow.log_param("anthropic_model", "claude-haiku-4-5-20251001")
    mlflow.log_param("gemini_model", "gemini-2.5-flash")

    print("Evaluating...")
    for i, sample in enumerate(samples):
        if i < len(results):
            continue  # already done in a previous (interrupted) run

        article   = sample["article"]
        reference = sample["highlights"]

        oai_summary, oai_lat = summarize_openai(article)
        ant_summary, ant_lat = summarize_anthropic(article)
        gem_summary, gem_lat = summarize_gemini(article)

        oai_r1, oai_r2, oai_rL = compute_rouge(oai_summary, reference)
        ant_r1, ant_r2, ant_rL = compute_rouge(ant_summary, reference)
        gem_r1, gem_r2, gem_rL = compute_rouge(gem_summary, reference)

        results.append({
            "article_snippet":    article[:80],
            "reference_summary":  reference,
            "openai_summary":     oai_summary,
            "anthropic_summary":  ant_summary,
            "gemini_summary":     gem_summary,
            "openai_latency_s":   round(oai_lat, 3),
            "anthropic_latency_s":round(ant_lat, 3),
            "gemini_latency_s":   round(gem_lat, 3),
            "openai_rouge1":      round(oai_r1, 4),
            "anthropic_rouge1":   round(ant_r1, 4),
            "gemini_rouge1":      round(gem_r1, 4),
            "openai_rouge2":      round(oai_r2, 4),
            "anthropic_rouge2":   round(ant_r2, 4),
            "gemini_rouge2":      round(gem_r2, 4),
            "openai_rougeL":      round(oai_rL, 4),
            "anthropic_rougeL":   round(ant_rL, 4),
            "gemini_rougeL":      round(gem_rL, 4),
        })

        if (i + 1) % 20 == 0:
            pd.DataFrame(results).to_csv(csv_path, index=False)  # checkpoint

        if i % 10 == 0:
            print(f"Progress: {i}/100")

    # Aggregate — derived from `results`, so it's correct whether this run
    # processed everything fresh or resumed a partial checkpoint.
    df = pd.DataFrame(results)

    mlflow.log_metric("openai_avg_rouge1",      df["openai_rouge1"].mean())
    mlflow.log_metric("openai_avg_rouge2",      df["openai_rouge2"].mean())
    mlflow.log_metric("openai_avg_rougeL",      df["openai_rougeL"].mean())
    mlflow.log_metric("anthropic_avg_rouge1",   df["anthropic_rouge1"].mean())
    mlflow.log_metric("anthropic_avg_rouge2",   df["anthropic_rouge2"].mean())
    mlflow.log_metric("anthropic_avg_rougeL",   df["anthropic_rougeL"].mean())
    mlflow.log_metric("gemini_avg_rouge1",      df["gemini_rouge1"].mean())
    mlflow.log_metric("gemini_avg_rouge2",      df["gemini_rouge2"].mean())
    mlflow.log_metric("gemini_avg_rougeL",      df["gemini_rougeL"].mean())
    mlflow.log_metric("openai_avg_latency_s",   df["openai_latency_s"].mean())
    mlflow.log_metric("anthropic_avg_latency_s",df["anthropic_latency_s"].mean())
    mlflow.log_metric("gemini_avg_latency_s",   df["gemini_latency_s"].mean())

    print(f"\n{'='*55}")
    print(f"{'Metric':<22} {'GPT-4o':>8} {'Haiku':>10} {'Gemini':>10}")
    print(f"{'='*55}")
    print(f"{'Avg Latency (s)':<22} {df['openai_latency_s'].mean():>8.3f} {df['anthropic_latency_s'].mean():>10.3f} {df['gemini_latency_s'].mean():>10.3f}")
    print(f"{'ROUGE-1':<22} {df['openai_rouge1'].mean():>8.3f} {df['anthropic_rouge1'].mean():>10.3f} {df['gemini_rouge1'].mean():>10.3f}")
    print(f"{'ROUGE-2':<22} {df['openai_rouge2'].mean():>8.3f} {df['anthropic_rouge2'].mean():>10.3f} {df['gemini_rouge2'].mean():>10.3f}")
    print(f"{'ROUGE-L':<22} {df['openai_rougeL'].mean():>8.3f} {df['anthropic_rougeL'].mean():>10.3f} {df['gemini_rougeL'].mean():>10.3f}")
    print(f"{'='*55}")

pd.DataFrame(results).to_csv(csv_path, index=False)  # final save
print(f"Saved to {csv_path}")


Loading CNN/DailyMail dataset...
Evaluating...
Progress: 0/100
Progress: 10/100
Progress: 20/100
Progress: 30/100
Progress: 40/100
Progress: 50/100
Progress: 60/100
Progress: 70/100
Progress: 80/100
Progress: 90/100

Metric                   GPT-4o      Haiku     Gemini
Avg Latency (s)           2.373      2.042      1.107
ROUGE-1                   0.323      0.309      0.324
ROUGE-2                   0.115      0.107      0.117
ROUGE-L                   0.221      0.209      0.222
Saved to ../logs/cnn_dailymail_baseline_20260526_134255.csv
